In [35]:
# 0. CILLM API Key 設定
# Notebook 預設使用 config.yaml 裡的 cillm_test provider：openai/gpt-oss-120b
# 請在執行本 cell 時貼上自己的 CILLM API Key；輸入內容不會顯示在 output。

import os
from getpass import getpass

if not os.getenv("CILLM_API_KEY"):
    os.environ["CILLM_API_KEY"] = getpass("請輸入 CILLM API Key: ").strip()

if os.getenv("CILLM_API_KEY"):
    print("CILLM_API_KEY 已設定，本 notebook 將使用 config.yaml 的 cillm_test provider。")
else:
    print("尚未設定 CILLM_API_KEY；後續 load_config() 會提示缺少 API key。")

請輸入 CILLM API Key:  ········


CILLM_API_KEY 已設定，本 notebook 將使用 config.yaml 的 cillm_test provider。


In [30]:
# 載入 packages, tools, parameters

import importlib
import llm_client
import trace_utils
import agent_prompts
import load_skills
import skill_tools
import skill_models

print("import llm and skill utils - stage 1")

importlib.reload(llm_client)
importlib.reload(trace_utils)
importlib.reload(agent_prompts)
importlib.reload(load_skills)
importlib.reload(skill_tools)
importlib.reload(skill_models)

from llm_client import load_config, create_client, call_llm
from trace_utils import setup_tracer, TokenTracker, trace_system # opentelemetry trace tools
from agent_prompts import build_context_route_messages, build_hint_messages, build_resource_route_messages, build_context_builder_messages, build_responder_messages
from load_skills import load_one_skill_metadata, load_skill_metadata, get_skill_by_id, load_full_skill, load_skill_reference, format_skill_metadata_for_prompt
from skill_tools import run_skill_script
from skill_models import ContextRouteResult, HintResult, ResourceRouteResult, ContextBuilderResult, OUT_OF_SCOPE_MESSAGE

print("import llm and skill utils - stage 2")

config, provider_config, api_key = load_config()
client = create_client(provider_config, api_key)
tracer = setup_tracer()

provider_name = config["provider"]
model = provider_config["model"]

generation = config["generation"]
parameters = {
    "model": model,
    "temperature": generation["temperature"],
    "max_tokens": generation["max_tokens"],
    "presence_penalty": generation["presence_penalty"],
}

print(f"Model Provider: {provider_name}")
print(f"Model Name: {model}")

import llm and skill utils - stage 1
import llm and skill utils - stage 2
Model Provider: openai
Model Name: gpt-4o


In [31]:
# 1. 僅呼叫 LLM，不使用 skill

memory = [
    {
        "role": "system",
        "content": "你是一個專業、有禮貌且樂於助人的助理，除了專有名詞，請用繁體中文回答 user 的提問與對話。以'您'為稱呼去稱呼 user。",
    }
]
token_tracker = TokenTracker()

print("Key in 'quit' while you want to end the chat.")
print("assistant: 您好，很高興見到您。今天我能夠如何幫助您呢？您有任何問題需要協助嗎？")
print("=" * 100)

while True:
    user_query = input("\nuser: ").strip()

    if user_query.lower() in ["q", "quit"]:
        print("system off")
        break

    if not user_query:
        continue

    memory.append({"role": "user", "content": user_query})
    token_tracker.start_batch()

    # trace
    response = call_llm(client, tracer, memory, parameters, config,token_tracker=token_tracker)
    
    token_tracker.print_usage()

    if response:
        print("="*100)
        print(f"user: {user_query}")
        print(f"assistant: {response}")
        print("=" * 100)
        memory.append({"role": "assistant", "content": response})

Key in 'quit' while you want to end the chat.
assistant: 您好，很高興見到您。今天我能夠如何幫助您呢？您有任何問題需要協助嗎？



user:  你好



Node_name: gpt-4o
  Trace:
    duration: 1.2437 秒
    status: OK

  Attributes:
    provider: openai
    model: gpt-4o
    error_message: 
    error_type: 
    llm_token: 74
batch_token: 74
total_token: 74
user: 你好
assistant: 您好！有什麼我可以幫助您的嗎？



user:  quit


system off


In [33]:
# 2. 單一 Skill

# load skill
selected_skill_id = "hr-training-validity-with-scripts" # 可改成 hr-free-ticket # hr-encourage-chinese # invoice-process-with-refs
selected_skill = load_one_skill_metadata(selected_skill_id) # 這個動作就像是在 Claude Code, Codex 裡面 /指定skill名稱 
skill_metadata = format_skill_metadata_for_prompt([selected_skill])
full_skill = load_full_skill(selected_skill)
index_key = next((key for key in selected_skill["references"] if key.endswith("policy_index")), None)
resource_index = load_skill_reference(selected_skill, selected_skill["references"][index_key]["path"]) if index_key else ""

token_tracker = TokenTracker()
agent_parameters = {**parameters, "temperature": 0} # 盡可能降低 node 判定的隨機性，responder 保留回覆語句自然

print(f"Skill list: {selected_skill_id}")
print("Key in 'quit' while you want to end the chat.")
print("assistant: 您好，很高興見到您。我可以協助您解答與公司休假規定，根據公司文件進行專業回覆。今天我能夠如何幫助您呢？您有任何問題需要協助嗎？")
print("=" * 100)

while True:
    user_query = input("\nuser: ").strip()

    if user_query.lower() in ["q", "quit"]:
        print("system off")
        break

    if not user_query:
        continue

    with trace_system(tracer, token_tracker, provider_name, model):
        
        # hint: 判斷 query 是否涉及本系統服務範圍
        hint_messages = build_hint_messages(user_query, skill_metadata)
        hint_result = call_llm(client, tracer, hint_messages, agent_parameters, config, node_name="hint", token_tracker=token_tracker, response_format={"type": "json_object"}, result_model=HintResult)

        if not hint_result.scope:
            response = OUT_OF_SCOPE_MESSAGE

        else: 
            # resource_router: 判斷需要讀取哪些 reference，以及是否需要執行 script
            resource_messages = build_resource_route_messages(user_query, full_skill, resource_index, selected_skill["scripts"])
            resource_result = call_llm(client, tracer, resource_messages, agent_parameters, config, node_name="resource_router", token_tracker=token_tracker, response_format={"type": "json_object"}, result_model=ResourceRouteResult)
            reference_contexts = [
                {"path": path, "content": load_skill_reference(selected_skill, path)}
                for path in resource_result.reference_paths
            ]
            script_results = [
                {"script_id": call.script_id, "result": run_skill_script(selected_skill, call.script_id, call.arguments)}
                for call in resource_result.script_calls
            ]

            # Context Builder: 根據 User Query、SKILL.md、政策內容與 script 結果萃取回答所需的 selected_context
            context_messages = build_context_builder_messages(user_query, selected_skill["skill_id"], full_skill, reference_contexts, script_results)
            context_result = call_llm(client, tracer, context_messages, agent_parameters, config, node_name="context_builder", token_tracker=token_tracker, response_format={"type": "json_object"}, result_model=ContextBuilderResult)

            # responder: 收到 selected_context 組織最終回覆
            responder_messages = build_responder_messages(user_query, context_result)
            response = call_llm(client, tracer, responder_messages, parameters, config, node_name="responder", token_tracker=token_tracker)

    token_tracker.print_usage()

    if response:
        print("=" * 100)
        print(f"user: {user_query}")
        print(f"assistant: {response}")
        print("=" * 100)

Skill list: hr-training-validity-with-scripts
Key in 'quit' while you want to end the chat.
assistant: 您好，很高興見到您。我可以協助您解答與公司休假規定，根據公司文件進行專業回覆。今天我能夠如何幫助您呢？您有任何問題需要協助嗎？



user:  我 2024/01/01 完成航空保安訓練，以 2026/06/24 來看是不是過期了？



Node_name: hint
  Trace:
    duration: 3.2079 秒
    status: OK

  Attributes:
    provider: openai
    model: gpt-4o
    error_message: 
    error_type: 
    llm_token: 409

  Evaluate:
    service_check: True
    skill_selection: hr-training-validity-with-scripts
    reason: 用戶詢問航空保安訓練的有效性，符合技能描述中關於檢查培訓或證書有效性的範疇。

Node_name: resource_router
  Trace:
    duration: 3.6066 秒
    status: OK

  Attributes:
    provider: openai
    model: gpt-4o
    error_message: 
    error_type: 
    llm_token: 1781

  Evaluate:
    references_list: ["references/training_policy_index.md"]
    script: [{"script_id": "training_validity_lookup", "arguments": {"training_type": "aviation_security", "completion_date": "2024-01-01", "as_of_date": "2026-06-24"}}]
    reason: 使用者詢問航空保安訓練的有效性，需計算日期，因此選擇相關 script 進行查詢。

Node_name: context_builder
  Trace:
    duration: 5.0813 秒
    status: OK

  Attributes:
    provider: openai
    model: gpt-4o
    error_message: 
    error_type: 
    llm_token: 1864

  Evaluate:



user:  quit


system off


In [ ]:
# 3. 多 Skill

# load skill
skills = load_skill_metadata() # 撈出所有 SKILL.md 的 metadata (name and description)
skills = [
    skill for skill in skills
    if "without" not in skill["skill_id"] and skill["skill_id"] != "hr-encourage-chinese"
] # 排除 without 對照組與中文鼓勵 skill，避免 hint 選到這些測試用 skill
skill_ids = [skill["skill_id"] for skill in skills] # skill list
skill_metadata = format_skill_metadata_for_prompt(skills)

token_tracker = TokenTracker()
agent_parameters = {**parameters, "temperature": 0}

print(f"Skill list: {skill_ids}")
print("Key in 'quit' while you want to end the chat.")
print("assistant: 您好，很高興見到您。我可以協助您查詢公司休假與員工優待機票規定。今天我能夠如何幫助您呢？您有任何問題需要協助嗎？")
print("=" * 100)

while True:
    user_query = input("\nuser: ").strip()

    if user_query.lower() in ["q", "quit"]:
        print("system off")
        break

    if not user_query:
        continue

    with trace_system(tracer, token_tracker, provider_name, model):
        
        # hint: 判斷服務範圍，並從所有 skills 中選出單一 skill
        hint_messages = build_hint_messages(user_query, skill_metadata)
        hint_result = call_llm(client, tracer, hint_messages, agent_parameters, config, node_name="hint", token_tracker=token_tracker, response_format={"type": "json_object"}, result_model=HintResult)

        if not hint_result.scope:
            response = OUT_OF_SCOPE_MESSAGE

        else:
            selected_skill = get_skill_by_id(hint_result.skill_id, skills)
            if selected_skill is None:
                raise ValueError("Skill not exsit.")

            full_skill = load_full_skill(selected_skill)
            index_key = next((key for key in selected_skill["references"] if key.endswith("policy_index")), None)
            resource_index = load_skill_reference(selected_skill, selected_skill["references"][index_key]["path"]) if index_key else ""

            # resource_router: 判斷需要讀取哪些 reference，以及是否需要執行 script
            resource_messages = build_resource_route_messages(user_query, full_skill, resource_index, selected_skill["scripts"])
            resource_result = call_llm(client, tracer, resource_messages, agent_parameters, config, node_name="resource_router", token_tracker=token_tracker, response_format={"type": "json_object"}, result_model=ResourceRouteResult)
            reference_contexts = [
                {"path": path, "content": load_skill_reference(selected_skill, path)}
                for path in resource_result.reference_paths
            ]
            script_results = [
                {"script_id": call.script_id, "result": run_skill_script(selected_skill, call.script_id, call.arguments)}
                for call in resource_result.script_calls
            ]

            # Context Builder: 根據 User Query、SKILL.md、政策內容與 script 結果萃取回答所需的 selected_context
            context_messages = build_context_builder_messages(user_query, selected_skill["skill_id"], full_skill, reference_contexts, script_results)
            context_result = call_llm(client, tracer, context_messages, agent_parameters, config, node_name="context_builder", token_tracker=token_tracker, response_format={"type": "json_object"}, result_model=ContextBuilderResult)

            # responder: 收到 selected_context 組織最終回覆
            responder_messages = build_responder_messages(user_query, context_result)
            response = call_llm(client, tracer, responder_messages, parameters, config, node_name="responder", token_tracker=token_tracker)

    token_tracker.print_usage()

    if response:
        print("=" * 100)
        print(f"user: {user_query}")
        print(f"assistant: {response}")
        print("=" * 100)

Skill list: ['hr-encourage-english', 'hr-free-ticket', 'hr-leave', 'hr-training-validity-with-scripts', 'invoice-process-with-refs']
Key in 'quit' while you want to end the chat.
assistant: 您好，很高興見到您。我可以協助您查詢公司休假與員工優待機票規定。今天我能夠如何幫助您呢？您有任何問題需要協助嗎？



user:  我需要鼓勵！！！！



Node_name: hint
  Trace:
    duration: 1.9734 秒
    status: OK

  Attributes:
    provider: openai
    model: gpt-4o
    error_message: 
    error_type: 
    llm_token: 843

  Evaluate:
    service_check: True
    skill_selection: hr-encourage-english
    reason: 用戶明確要求鼓勵，這符合 hr-encourage-english 技能的服務範圍。

Node_name: resource_router
  Trace:
    duration: 2.4516 秒
    status: OK

  Attributes:
    provider: openai
    model: gpt-4o
    error_message: 
    error_type: 
    llm_token: 1456

  Evaluate:
    references_list: []
    script: 
    reason: The user is asking for encouragement, which matches the purpose of the hr-encourage-english skill. No additional resources or scripts are needed to provide a supportive response.

Node_name: context_builder
  Trace:
    duration: 4.0671 秒
    status: OK

  Attributes:
    provider: openai
    model: gpt-4o
    error_message: 
    error_type: 
    llm_token: 1473

  Evaluate:
    skill_id: hr-encourage-english
    information_complete: True



user:  好，我 2024/01/01 完成航空保安訓練，以 2026/06/24 來看是不是過期了？



Node_name: hint
  Trace:
    duration: 2.6404 秒
    status: OK

  Attributes:
    provider: openai
    model: gpt-4o
    error_message: 
    error_type: 
    llm_token: 880

  Evaluate:
    service_check: True
    skill_selection: hr-training-validity-with-scripts
    reason: 用戶詢問航空保安訓練的完成日期和有效性，這符合 hr-training-validity-with-scripts 的服務範圍。

Node_name: resource_router
  Trace:
    duration: 3.9712 秒
    status: OK

  Attributes:
    provider: openai
    model: gpt-4o
    error_message: 
    error_type: 
    llm_token: 1776

  Evaluate:
    references_list: []
    script: [{"script_id": "training_validity_lookup", "arguments": {"training_type": "aviation_security", "completion_date": "2024-01-01", "as_of_date": "2026-06-24"}}]
    reason: 使用者詢問航空保安訓練的有效性，提供了完成日期和檢查日期，需要使用腳本計算過期狀態。

Node_name: context_builder
  Trace:
    duration: 6.3583 秒
    status: OK

  Attributes:
    provider: openai
    model: gpt-4o
    error_message: 
    error_type: 
    llm_token: 1296

  Evaluate:
    skill_